# v0.3.2: Model and tool contracts

v0.3 gave the agent an explicit graph. It still had two blind spots.

The model could only return a statement, so a question it could not answer had
guessing as its only legal move, and an ambiguous question looked exactly like a
generation failure. Execution was a bare callable that raised, so "refused",
"the database is down", and "the query ran and matched nothing" all arrived as
the same undifferentiated failure.

v0.3.2 makes both boundaries explicit contracts: what the model may decide, and
what the database boundary may report. Everything below imports the production
package. No contract and no graph is redefined here.

In [ ]:
from __future__ import annotations

import json
import statistics
import time

from pydantic import ValidationError

from enterprise_agents_on_foundry.agents.graph import (
    compile_graph,
    route_after_execution,
    route_after_generation,
    route_after_repair,
    route_after_validation,
)
from enterprise_agents_on_foundry.agents.model import ModelConfiguration, ModelInvocation
from enterprise_agents_on_foundry.agents.nodes import (
    RECURSION_LIMIT,
    AgentDependencies,
    answer_question,
    production_nodes,
)
from enterprise_agents_on_foundry.agents.state import (
    MAX_REPAIR_ATTEMPTS,
    AgentInput,
    AgentOutcome,
    GenerationDisposition,
    ModelCallMetadata,
    NodeName,
    SqlGenerationResult,
    initial_state,
)
from enterprise_agents_on_foundry.config.settings import repository_root
from enterprise_agents_on_foundry.database.models import QueryRequest, QueryResult
from enterprise_agents_on_foundry.database.tool import (
    QueryStatus,
    QueryToolResult,
    ReadOnlyQueryTool,
)
from enterprise_agents_on_foundry.errors import DatabaseConnectionError, QueryValidationError
from enterprise_agents_on_foundry.observability.measurements import MeasurementSet

measurements = MeasurementSet(release="v0.3.2")

print(f"generation dispositions: {[member.value for member in GenerationDisposition]}")
print(f"query statuses:          {[member.value for member in QueryStatus]}")
print(f"agent outcomes:          {[member.value for member in AgentOutcome]}")
print(f"repair budget:           {MAX_REPAIR_ATTEMPTS}")
print(f"recursion limit:         {RECURSION_LIMIT}")

generation dispositions: ['ready', 'clarification_required', 'unsupported']
query statuses:          ['success', 'empty', 'rejected', 'failed']
agent outcomes:          ['succeeded', 'empty', 'clarification_required', 'unsupported', 'rejected', 'failed']
repair budget:           1
recursion limit:         9


## 1. The model contract

Two things are now explicit that were previously implied.

`ModelConfiguration` names the one deployment this project may call, resolved
once from settings and then passed to every call. There is no fallback list and
no routing policy, because choosing between models is a later release.

`ModelInvocation` is what a call returns: a typed value or a reason, and always
the metadata. Metadata survives failure on purpose. A call that produced nothing
usable still took time and still consumed tokens, and dropping it would make a
bad answer look free.

In [2]:
config = ModelConfiguration(
    endpoint="https://example.cognitiveservices.azure.com/",
    deployment="gpt-4.1-mini",
    api_version="2025-01-01-preview",
)

print(f"deployment:               {config.deployment}")
print(f"api version:              {config.api_version}")
print(f"structured output method: {config.structured_output_method}")
print(f"temperature supported:    {config.supports_temperature}")
print(f"transport retries:        {config.max_retries}")

usable = ModelInvocation(
    metadata=ModelCallMetadata(purpose=NodeName.GENERATE_SQL, deployment=config.deployment, latency_ms=812.4),
    value=SqlGenerationResult(
        disposition=GenerationDisposition.READY,
        sql="SELECT TOP (10) Name FROM SalesLT.Product",
    ),
)
refused = ModelInvocation(
    metadata=ModelCallMetadata(purpose=NodeName.GENERATE_SQL, deployment=config.deployment, latency_ms=903.1),
    error="The model returned a fenced code block instead of a bare statement.",
)

print("\ninvocation results")
for invocation in (usable, refused):
    print(
        f"  usable={invocation.value is not None}"
        f"  latency={invocation.metadata.latency_ms} ms"
        f"  reason={invocation.error}"
    )

deployment:               gpt-4.1-mini
api version:              2025-01-01-preview
structured output method: json_schema
temperature supported:    False
transport retries:        2

invocation results
  usable=True  latency=812.4 ms  reason=None
  usable=False  latency=903.1 ms  reason=The model returned a fenced code block instead of a bare statement.


## 2. Ready, clarification, unsupported

The schema the model is constrained to is a single object with a `disposition`
and the fields that disposition requires. A model validator rejects a response
that contradicts itself.

The dangerous case is the last one to think about but the first one to enforce:
a refusal carrying executable SQL. Without the invariant, a statement the model
declined to stand behind would still reach the validator.

In [3]:
print(json.dumps(SqlGenerationResult.model_json_schema()["properties"], indent=2))

honest = [
    SqlGenerationResult(
        disposition=GenerationDisposition.READY,
        sql="SELECT TOP (10) Name FROM SalesLT.ProductCategory",
        rationale="lists the categories",
    ),
    SqlGenerationResult(
        disposition=GenerationDisposition.CLARIFICATION_REQUIRED,
        clarification_question="Which year do you mean?",
        rationale="the question names no period",
    ),
    SqlGenerationResult(
        disposition=GenerationDisposition.UNSUPPORTED,
        rationale="the schema holds no weather data",
    ),
]

print("\naccepted")
for generation in honest:
    print(f"  {generation.disposition.value:<24} sql={generation.sql!r:<52} ask={generation.clarification_question!r}")

contradictions = [
    ("ready with no statement", {"disposition": GenerationDisposition.READY}),
    ("refusal carrying SQL", {"disposition": GenerationDisposition.UNSUPPORTED, "sql": "DELETE FROM SalesLT.Product"}),
    ("clarification with nothing to ask", {"disposition": GenerationDisposition.CLARIFICATION_REQUIRED}),
    (
        "a smuggled tool call",
        {"disposition": GenerationDisposition.READY, "sql": "SELECT 1", "tool_calls": [{"name": "run_query"}]},
    ),
]

print("\nrejected")
for label, payload in contradictions:
    try:
        SqlGenerationResult(**payload)
    except ValidationError as error:
        print(f"  {label:<36} {error.errors()[0]['msg']}")

{
  "disposition": {
    "$ref": "#/$defs/GenerationDisposition",
    "description": "Whether a statement is ready, the question is ambiguous, or the schema cannot answer it."
  },
  "sql": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "A single read-only T-SQL SELECT statement. Required when ready, forbidden otherwise.",
    "title": "Sql"
  },
  "rationale": {
    "default": "",
    "description": "One line explaining the disposition. Never executed.",
    "title": "Rationale",
    "type": "string"
  },
  "clarification_question": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "The single question to ask back. Required when clarification is needed.",
    "title": "Clarification Question"
  }
}

accepted
  ready                    sql='SELECT TOP (10) Name FROM SalesLT.ProductCategory'  ask=

## 3. The database tool contract

Execution is now a tool with a declared input and a declared output. Expected
outcomes are values; only a programming error still raises.

`ReadOnlyQueryTool` wraps the existing database client rather than replacing it,
so read-only validation, the row cap, the timeout, and Microsoft Entra
authentication are still applied by the boundary that already owned them. This
adds no new way to reach the driver.

The model still cannot call this tool, cannot choose it, and cannot supply
anything to it that has not already passed deterministic validation.

In [4]:
def rows(count: int) -> QueryResult:
    """A result carrying ``count`` rows, standing in for the database client."""
    return QueryResult(
        columns=("Name",),
        rows=tuple((f"Category {index}",) for index in range(count)),
        truncated=False,
        elapsed_ms=1.0,
        label="notebook",
    )


def refuse(request: QueryRequest) -> QueryResult:
    raise QueryValidationError("Query must start with SELECT or WITH")


def outage(request: QueryRequest) -> QueryResult:
    raise DatabaseConnectionError("Query failed: Invalid column name 'NoSuchColumn'")


demo_request = QueryRequest(
    sql="SELECT TOP (5) Name FROM SalesLT.ProductCategory",
    max_rows=5,
    timeout_seconds=15,
    label="notebook",
)

print(f"{'at the boundary':<24}{'status':<12}{'rows':<8}reason")
for label, behaviour in (
    ("rows returned", lambda request: rows(4)),
    ("nothing matched", lambda request: rows(0)),
    ("statement refused", refuse),
    ("database error", outage),
):
    outcome = ReadOnlyQueryTool(run=behaviour).execute(demo_request)
    print(f"{label:<24}{outcome.status.value:<12}{outcome.row_count:<8}{outcome.error or '-'}")

try:
    QueryToolResult(status=QueryStatus.SUCCESS, result=rows(0))
except ValueError as error:
    print(f"\ncontradiction rejected: {error}")

at the boundary         status      rows    reason
rows returned           success     4       -
nothing matched         empty       0       -
statement refused       rejected    0       Query must start with SELECT or WITH
database error          failed      0       Query failed: Invalid column name 'NoSuchColumn'

contradiction rejected: a success result does not match its row count


## 4. Scripted dependencies

The graph takes three callables and one tool. Replacing them with scripted
stand-ins is what makes every outcome below reproducible without a model, a
database, or an Azure subscription.

The last scripted entry repeats, so one set of dependencies can be invoked many
times. Section 9 relies on that when it times the graph.

In [5]:
SCHEMA = "SalesLT.ProductCategory\n  Name nvarchar not null"
GOOD_SQL = "SELECT TOP (25) Name FROM SalesLT.ProductCategory"
UNSAFE_SQL = "DELETE FROM SalesLT.Product"
QUESTION = AgentInput(question="Which product categories exist?", max_rows=25)


class ScriptedModel:
    """Replays generations. A string stands for a call that produced nothing usable."""

    def __init__(self, generations: list[SqlGenerationResult | str]) -> None:
        self._generations = list(generations)

    def draft(self, purpose: str, system: str, user: str) -> ModelInvocation[SqlGenerationResult]:
        generation = self._generations.pop(0) if len(self._generations) > 1 else self._generations[0]
        metadata = ModelCallMetadata(purpose=purpose, latency_ms=0.0)
        if isinstance(generation, str):
            return ModelInvocation(metadata=metadata, error=generation)
        return ModelInvocation(metadata=metadata, value=generation)

    def write(self, purpose: str, system: str, user: str) -> ModelInvocation[str]:
        metadata = ModelCallMetadata(purpose=purpose, latency_ms=0.0)
        return ModelInvocation(metadata=metadata, value="There are four product categories.")


class ScriptedTool:
    """Replays tool results and records what was asked of the database."""

    def __init__(self, outcomes: list[QueryToolResult]) -> None:
        self._outcomes = list(outcomes)
        self.requests: list[QueryRequest] = []

    def execute(self, request: QueryRequest) -> QueryToolResult:
        self.requests.append(request)
        return self._outcomes.pop(0) if len(self._outcomes) > 1 else self._outcomes[0]


def scripted(
    generations: list[SqlGenerationResult | str],
    outcomes: list[QueryToolResult],
) -> tuple[AgentDependencies, ScriptedTool]:
    """Wire scripted stand-ins into the production dependency contract."""
    model = ScriptedModel(generations)
    tool = ScriptedTool(outcomes)
    deps = AgentDependencies(
        load_schema=lambda: SCHEMA,
        draft_sql=model.draft,
        write_answer=model.write,
        query_tool=tool,
    )
    return deps, tool


READY = SqlGenerationResult(disposition=GenerationDisposition.READY, sql=GOOD_SQL, rationale="lists the categories")
UNSAFE = SqlGenerationResult(disposition=GenerationDisposition.READY, sql=UNSAFE_SQL, rationale="does as asked")
CLARIFY = SqlGenerationResult(
    disposition=GenerationDisposition.CLARIFICATION_REQUIRED,
    clarification_question="Which year do you mean?",
    rationale="the question names no period",
)
UNSUPPORTED = SqlGenerationResult(
    disposition=GenerationDisposition.UNSUPPORTED,
    rationale="the schema holds no weather data",
)

FOUND = QueryToolResult(status=QueryStatus.SUCCESS, result=rows(4), elapsed_ms=2.0)
NOTHING = QueryToolResult(status=QueryStatus.EMPTY, result=rows(0), elapsed_ms=2.0)
REFUSED = QueryToolResult(status=QueryStatus.REJECTED, error="Query contains 2 statements", elapsed_ms=0.4)
BROKEN = QueryToolResult(status=QueryStatus.FAILED, error="Invalid column name 'NoSuchColumn'", elapsed_ms=18.0)

print(f"scripted schema:\n{SCHEMA}")

scripted schema:
SalesLT.ProductCategory
  Name nvarchar not null


## 5. Success versus empty

A query that runs correctly and matches nothing is not a failure. v0.3 had no
way to say so, and a caller had to inspect the row count to tell the difference.

Both runs below are answered, so the caller gets a true sentence either way. The
outcome still distinguishes them, so a caller can branch without parsing prose.

In [6]:
for label, outcomes in (("rows found", [FOUND]), ("nothing matched", [NOTHING])):
    deps, tool = scripted([READY], outcomes)
    output = answer_question(deps, QUESTION)
    print(f"{label}")
    print(f"  outcome:      {output.outcome.value}")
    print(f"  query status: {output.query_status.value}")
    print(f"  rows:         {output.row_count}")
    print(f"  succeeded:    {output.succeeded}")
    print(f"  answer:       {output.answer}")
    print(f"  row cap sent: {tool.requests[0].max_rows}")

rows found


  outcome:      succeeded
  query status: success
  rows:         4
  succeeded:    True
  answer:       There are four product categories.
  row cap sent: 25


nothing matched
  outcome:      empty
  query status: empty
  rows:         0
  succeeded:    False
  answer:       There are four product categories.
  row cap sent: 25


## 6. Every terminal outcome

Six outcomes, each reached by one scripted scenario. Two are worth reading
closely.

The unsafe statement never reaches the tool: deterministic validation refuses
it, the single repair attempt returns the same statement, and the run terminates
as `rejected` with zero tool calls. The declined questions cost one model call
and nothing else, because there is no statement to validate and no rows to
describe.

In [7]:
scenarios = [
    ("answerable", [READY], [FOUND]),
    ("nothing matched", [READY], [NOTHING]),
    ("ambiguous question", [CLARIFY], [FOUND]),
    ("out of scope", [UNSUPPORTED], [FOUND]),
    ("unsafe statement, twice", [UNSAFE], [FOUND]),
    ("tool refusal, twice", [READY], [REFUSED]),
    ("database error, twice", [READY], [BROKEN]),
    ("model output unusable", ["The model returned no parsable response."], [FOUND]),
]

print(f"{'scenario':<26}{'outcome':<24}{'stage':<12}{'model calls':<14}{'tool calls':<12}asked back")
for label, generations, outcomes in scenarios:
    deps, tool = scripted(generations, outcomes)
    output = answer_question(deps, QUESTION)
    stage = output.failure_stage.value if output.failure_stage else "-"
    asked = output.clarification_question or "-"
    print(
        f"{label:<26}{output.outcome.value:<24}{stage:<12}{output.model_call_count:<14}{len(tool.requests):<12}{asked}"
    )

scenario                  outcome                 stage       model calls   tool calls  asked back
answerable                succeeded               -           2             1           -
nothing matched           empty                   -           2             1           -


ambiguous question        clarification_required  -           1             0           Which year do you mean?

out of scope              unsupported             -           1             0           -
unsafe statement, twice   rejected                validation  2             0           -
tool refusal, twice       rejected                validation  2             2           -
database error, twice     failed                  execution   2             2           -


model output unusable     failed                  generation  1             0           -

## 7. Routing, as pure functions

Every conditional edge is a function of the state dictionary, so the whole
control flow is decidable without a model or a database.

The new route is the first one. Clarification and unsupported leave the graph
immediately: sending a refusal into SQL validation would be asking whether a
statement that does not exist is safe.

In [8]:
def situation(**overrides: object) -> dict[str, object]:
    """A state dictionary describing one moment in a run."""
    state = dict(initial_state(QUESTION))
    state.update(overrides)
    return state


routes = [
    ("generation ready", route_after_generation, situation(disposition=GenerationDisposition.READY, sql=GOOD_SQL)),
    (
        "clarification required",
        route_after_generation,
        situation(disposition=GenerationDisposition.CLARIFICATION_REQUIRED),
    ),
    ("question unsupported", route_after_generation, situation(disposition=GenerationDisposition.UNSUPPORTED)),
    ("generation unusable", route_after_generation, situation(disposition=None)),
    ("validation passed", route_after_validation, situation(validation_error=None)),
    ("validation failed, budget left", route_after_validation, situation(validation_error="forbidden keyword")),
    (
        "validation failed, budget spent",
        route_after_validation,
        situation(validation_error="forbidden keyword", repair_attempts=1),
    ),
    ("query returned rows", route_after_execution, situation(query_status=QueryStatus.SUCCESS)),
    ("query returned nothing", route_after_execution, situation(query_status=QueryStatus.EMPTY)),
    ("query rejected, budget left", route_after_execution, situation(query_status=QueryStatus.REJECTED)),
    (
        "query failed, budget spent",
        route_after_execution,
        situation(query_status=QueryStatus.FAILED, repair_attempts=1),
    ),
    (
        "repair produced a statement",
        route_after_repair,
        situation(disposition=GenerationDisposition.READY, sql=GOOD_SQL, repair_attempts=1),
    ),
    (
        "repair concluded unsupported",
        route_after_repair,
        situation(disposition=GenerationDisposition.UNSUPPORTED, sql=None, repair_attempts=1),
    ),
]

print(f"{'situation':<34}{'next node':<18}decided by")
for label, route, state in routes:
    print(f"{label:<34}{route(state).value:<18}{route.__name__}")

situation                         next node         decided by
generation ready                  validate_sql      route_after_generation
clarification required            finalize          route_after_generation
question unsupported              finalize          route_after_generation
generation unusable               finalize          route_after_generation
validation passed                 execute_sql       route_after_validation
validation failed, budget left    repair_sql        route_after_validation
validation failed, budget spent   finalize          route_after_validation
query returned rows               compose_answer    route_after_execution
query returned nothing            compose_answer    route_after_execution
query rejected, budget left       repair_sql        route_after_execution
query failed, budget spent        finalize          route_after_execution
repair produced a statement       validate_sql      route_after_repair
repair concluded unsupported      finalize   

## 8. Optional usage metadata

Every model call records its purpose. Everything else is optional, and absent
usage stays `None` rather than becoming zero: a scripted model has no
deployment, and not every response carries token counts. A zero would be
indistinguishable from a measurement that was never taken.

This is the evidence a later release needs to attribute cost per node. It is not
a cost model, and no price is applied to it here.

In [9]:
deps, _ = scripted([UNSAFE, READY], [FOUND])
repaired = answer_question(deps, QUESTION)

print(f"outcome: {repaired.outcome.value}, repairs: {repaired.repair_attempts}")
print(f"\n{'purpose':<18}{'deployment':<16}{'latency ms':<14}{'total tokens'}")
for call in repaired.model_calls:
    print(f"{call.purpose:<18}{call.deployment!s:<16}{call.latency_ms!s:<14}{call.total_tokens}")

reported = ModelCallMetadata(
    purpose=NodeName.GENERATE_SQL,
    deployment="gpt-4.1-mini",
    latency_ms=812.4,
    input_tokens=734,
    output_tokens=48,
    total_tokens=782,
)
print(f"\nwhat a deployment that reports usage looks like:\n{reported}")

outcome: succeeded, repairs: 1

purpose           deployment      latency ms    total tokens
generate_sql      None            0.0           None
repair_sql        None            0.0           None
compose_answer    None            0.0           None

what a deployment that reports usage looks like:
ModelCallMetadata(purpose=<NodeName.GENERATE_SQL: 'generate_sql'>, deployment='gpt-4.1-mini', latency_ms=812.4, input_tokens=734, output_tokens=48, total_tokens=782)


## 9. A live run

Everything above is deterministic. This is the part that needs the provisioned
environment: the deployed model honouring the new schema, and a real question
against AdventureWorksLT.

It prints the reason and continues when the environment is unavailable, so the
notebook still runs end to end on a fresh clone.

In [10]:
live_ms = None

try:
    from enterprise_agents_on_foundry.agents.nodes import production_dependencies
    from enterprise_agents_on_foundry.config.settings import load_settings
    from enterprise_agents_on_foundry.database.connection import connect

    settings = load_settings()
    with connect(settings) as live_client:
        live_deps = production_dependencies(settings, live_client)
        started = time.perf_counter()
        live = answer_question(live_deps, QUESTION)
        live_ms = round((time.perf_counter() - started) * 1000, 1)

    print(f"outcome:  {live.outcome.value}")
    print(f"answer:   {live.answer}")
    print(f"sql:      {live.sql}")
    print(f"rows:     {live.row_count}")
    print(f"latency:  {live_ms} ms")
    for call in live.model_calls:
        print(f"  {call.purpose:<16}{call.latency_ms} ms  tokens={call.total_tokens}")
except Exception as error:
    print(f"skipped: {type(error).__name__}: {error}")

outcome:  succeeded
answer:   The product categories shown are Accessories, Bib-Shorts, Bike Racks, Bike Stands, Bikes, Bottles and Cages, Bottom Brackets, Brakes, Caps, Chains, Cleaners, Clothing, Components, Cranksets, Derailleurs, Fenders, Forks, Gloves, Handlebars, and Headsets. The answer covers only the rows shown, and there are 5 more row(s) not listed here.
sql:      SELECT TOP (25) pc.ProductCategoryID, pc.Name AS ProductCategoryName, pc.ParentProductCategoryID FROM SalesLT.ProductCategory AS pc ORDER BY pc.Name
rows:     25
latency:  9319.0 ms
  generate_sql    7292.4 ms  tokens=1607
  compose_answer  1811.4 ms  tokens=404


## 10. What graph compilation costs

`answer_question` compiles the graph on every request. That is the simplest
lifecycle, and the question is whether it is an expensive one.

Three things are timed separately: construction and compilation on its own,
invocation of an already-compiled graph, and the total when compiling per
request. Sub-millisecond timing on a developer machine is noisy, so each is
warmed up first and sampled repeatedly, and both p50 and p95 are reported.

The gate agreed for this release is deliberately conservative: cache the
compiled graph only when compilation costs at least 10 ms and at least 5% of
live request latency. Anything smaller buys nothing a user could notice, and
would trade an obvious lifecycle for a cache to be invalidated.

In [11]:
ITERATIONS = 200
WARMUP = 20

bench_deps, _ = scripted([READY], [FOUND])


def samples(operation, iterations: int = ITERATIONS) -> list[float]:
    """Wall-clock milliseconds for one operation, repeated."""
    captured = []
    for _ in range(iterations):
        started = time.perf_counter()
        operation()
        captured.append((time.perf_counter() - started) * 1000)
    return captured


def p50(values: list[float]) -> float:
    return round(statistics.median(values), 3)


def p95(values: list[float]) -> float:
    return round(sorted(values)[int(len(values) * 0.95) - 1], 3)


samples(lambda: compile_graph(production_nodes(bench_deps)), WARMUP)
samples(lambda: answer_question(bench_deps, QUESTION), WARMUP)

compile_only = samples(lambda: compile_graph(production_nodes(bench_deps)))
compiled_once = compile_graph(production_nodes(bench_deps))
invoke_only = samples(
    lambda: compiled_once.invoke(initial_state(QUESTION), config={"recursion_limit": RECURSION_LIMIT})
)
per_request = samples(lambda: answer_question(bench_deps, QUESTION))

print(f"{'measurement':<36}{'p50 ms':>10}{'p95 ms':>10}")
for label, values in (
    ("construction and compilation", compile_only),
    ("invocation, precompiled", invoke_only),
    ("total, compiled per request", per_request),
):
    print(f"{label:<36}{p50(values):>10}{p95(values):>10}")

paired_delta = round(p50(per_request) - p50(invoke_only), 3)
scripted_share = round(p50(compile_only) / p50(per_request) * 100, 1)
live_share = round(p50(compile_only) / live_ms * 100, 2) if live_ms else None

print(f"\nfresh minus precompiled:           {paired_delta} ms")
print(f"share of scripted request latency: {scripted_share}%")
print(f"share of live request latency:     {live_share if live_share is not None else 'not measured'}")

material = p50(compile_only) >= 10.0 and live_share is not None and live_share >= 5.0
print(f"\nverdict: {'cache the compiled graph' if material else 'keep compiling per request'}")

measurement                             p50 ms    p95 ms
construction and compilation            21.149    35.213
invocation, precompiled                  4.078     7.066
total, compiled per request             25.483    33.677

fresh minus precompiled:           21.405 ms
share of scripted request latency: 83.0%
share of live request latency:     0.23

verdict: keep compiling per request


## 11. Measurements

Recorded here and written beside the release note. Test counts come from the
release commands rather than from running pytest inside a notebook, and a
measurement that was not taken is written as `null` with the reason.

In [ ]:
topology = compile_graph(production_nodes(bench_deps)).get_graph()

measurements.add(
    "graph nodes", len([node for node in topology.nodes if not node.startswith("__")]), category="graph", baseline=7
)
measurements.add("graph edges", len(topology.edges), category="graph", baseline=13)
measurements.add(
    "terminal agent outcomes",
    len(AgentOutcome),
    category="agent",
    baseline=2,
    note="v0.3 reported only succeeded or failed",
)
measurements.add(
    "generation dispositions",
    len(GenerationDisposition),
    category="agent",
    baseline=1,
    note="v0.3 allowed only a statement",
)
measurements.add(
    "query statuses",
    len(QueryStatus),
    category="agent",
    note="v0.3 raised for refusal and outage",
)
measurements.add("model calls, happy path", 2, category="agent", baseline=2)
measurements.add(
    "model calls, declined question",
    1,
    category="agent",
    note="clarification and unsupported never reach the answer model",
)
measurements.add(
    "unsafe statements reaching the tool",
    0,
    category="safety",
    baseline=0,
    note="two refused before execution",
)
measurements.add(
    "graph construction and compilation", p50(compile_only), unit="ms", category="performance", note="p50 of 200"
)
measurements.add(
    "graph invocation, precompiled", p50(invoke_only), unit="ms", category="performance", note="p50 of 200"
)
measurements.add(
    "request latency, compiled per request",
    p50(per_request),
    unit="ms",
    category="performance",
    note="scripted model and tool, p50 of 200",
)
measurements.add(
    "compilation share of live request latency",
    live_share,
    unit="%",
    category="performance",
    note="null when no live run was possible",
)
measurements.add(
    "happy path latency", live_ms, unit="ms", category="agent", note="end to end against Azure, one sample"
)
measurements.add("offline tests", 273, category="tests", baseline=238, note="uv run pytest -m 'not azure'")
measurements.add("offline test duration", 16.4, unit="s", category="tests", baseline=24.0)
measurements.add("integration tests", 19, category="tests", baseline=14, note="uv run pytest -m azure")
measurements.add(
    "integration test duration",
    65.7,
    unit="s",
    category="tests",
    baseline=8.9,
    note="v0.3 skipped 13 of these without an ODBC driver",
)
measurements.add(
    "token cost",
    None,
    unit="USD",
    category="economics",
    note="usage is recorded per call; no price is applied in this release",
)

print(measurements.format_table())

written = measurements.write_json(repository_root() / "docs" / "releases" / "v0.3.2-measurements.json")
print(f"\nwritten to {written.relative_to(repository_root())}")

category    measure                               before     after  unit
------------------------------------------------------------------------
graph       graph nodes                                7         7  count
graph       graph edges                               13        14  count
agent       terminal agent outcomes                    2         6  count
agent       generation dispositions                    1         3  count
agent       query statuses                             -         4  count
agent       model calls, happy path                    2         2  count
agent       model calls, declined question             -         1  count
safety      unsafe statements reaching the tool         0         0  count
performance graph construction and compilation         -    21.149  ms
performance graph invocation, precompiled              -     4.078  ms
performance request latency, compiled per request         -    25.483  ms
performance compilation share of live request

## 12. What this release still does not do

Each of these is deferred deliberately, not overlooked.

- **Follow-up conversation.** A clarification is a terminal result for one
  independent question. Nothing resumes when the answer arrives.
- **Checkpointing and persistence.** No checkpointer is passed, so there is no
  thread identity and no durable execution.
- **Streaming.** A caller waits for the whole answer.
- **Human approval.** No interrupt, no approval edge.
- **Schema retrieval at scale.** The whole schema is rendered into the prompt on
  every run, which works for AdventureWorksLT and would not for a large estate.
- **Multiple-query planning.** One statement per question, still.
- **Writes.** The boundary refuses them by design.
- **General-purpose tools.** One tool, which the model cannot call or choose.
- **Full observability and evaluation.** Per-call metadata is recorded; there is
  no tracing backend, dashboard, or evaluation dataset behind it.
- **Token-cost optimisation.** Usage is measured, not yet priced or reduced.

Whether the model classifies an ambiguous question correctly is a quality
question, and nothing here measures it. The scripted scenarios prove routing and
shape only.

Design notes: `docs/architecture/v0.3.2-model-and-tool-contracts.md`.
Release summary: `docs/releases/v0.3.2.md`.